## Data Loading

In [ ]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
from openpyxl import Workbook

# create once, pass around multiple times for writing metrics
wb = Workbook()

warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 2.17 s
Wall time: 2.63 s


In [ ]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3


# Sanity check: Using Iterative Multi-Label staratified sampling.



In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

def stratify_holdout_train_test_split(
    df,
    target_col: str = 'target',
    k_value_col: str = None,
    test_size: float = 0.1,
    random_state: int = 40
):
    """
    Produce a balanced hold-out split by iteratively stratifying on:
      - multi-label `target_col` (list of algorithms)
      - and optionally `k_value_col`.

    If `k_value_col` is None, stratifies only on `target_col`.

    Returns
    -------
    train_df, test_df : pd.DataFrame
    """
    
    df = df.copy()

    rows = []
    for _, row in df.iterrows():
        # copy only the feature columns
        new_row = row.copy()
        # find tied fastest algorithms
        times = [row[alg] for alg in algorithms]
        m = min(times)
        tied = [alg for alg in algorithms if row[alg] == m]
        # assign multi‐label target list
        new_row[target_col] = tied
        rows.append(new_row)

    df = pd.DataFrame(rows).reset_index(drop=True)

    # 1) Build multilabel indicator matrix for the target lists
    Y_alg = np.vstack([
        [int(alg in row) for alg in algorithms]
        for row in df[target_col]
    ])

    # 2) If k_value_col is given, one-hot encode it and combine
    if k_value_col is not None:
        unique_ks = sorted(df[k_value_col].unique())
        Y_k = np.vstack([
            [int(k == kv) for kv in unique_ks]
            for k in df[k_value_col]
        ])
        Y_multi = np.hstack([Y_alg, Y_k])
    else:
        Y_multi = Y_alg

    # 3) Stratified shuffle split on multilabel
    splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state
    )
    train_idx, test_idx = next(splitter.split(X=np.zeros((len(df), 1)), y=Y_multi))

    # 4) Select DataFrames
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df  = df.iloc[test_idx] .reset_index(drop=True)

    print("Train set shape:", train_df.shape)
    print("Test set shape: ", test_df.shape)

    return train_df, test_df



In [ ]:
def debug_split(train_df, test_df):
    print("TARGET distribution (train):")
    print(pd.Series([tuple(x) for x in train_df['target']]).value_counts(normalize=True))
    print("\nTARGET distribution (test):")
    print(pd.Series([tuple(x) for x in test_df['target']]).value_counts(normalize=True))

    print("\nK_VALUE distribution (train):")
    print(train_df['k_value'].value_counts(normalize=True))
    print("\nK_VALUE distribution (test):")
    print(test_df['k_value'].value_counts(normalize=True))

# Example:
train_df, test_df = stratify_holdout_train_test_split(cat_df, k_value_col='k_value')
debug_split(train_df, test_df)

train_df2, test_df2 = stratify_holdout_train_test_split(cat_df, k_value_col=None)
debug_split(train_df2, test_df2)
